# Two-Stage NH₃/H₂ Staged Combustion — PSR Sweep

## Run guide

Run cells from top to bottom. First confirm the single-case test converges; then launch the 605-case sweep. Later sections analyze the saved sweep, and the stricter-solver rerun is opt-in. The mechanism file must be available from the notebook working directory.

### Model workflow

**Fuel blend + primary air → primary PSR → secondary air → secondary PSR → diagnostics and analysis**

The sweep covers NH₃ fraction (0–100%), primary equivalence ratio (1.0–2.0), and residence time (20–100 ms), with overall equivalence ratio fixed at 0.9.


## 1. Imports

In [ ]:
import time
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cantera as ct

print("Cantera version:", ct.__version__)

from double_stage.staged_combustion import (
    AIR,
    MECH,
    PHI_OVERALL,
    P_IN,
    T_IN,
    calculate_air_amounts,
    integrate_to_steady_state,
)


## 2. Model settings

These values are fixed in the initial study. Fuel composition, primary φ, and residence time are varied later.

The NH₃/H₂ composition is specified on a mole basis.


In [ ]:
print("Mechanism:", MECH)
print("Inlet temperature:", T_IN, "K")
print("Inlet pressure:", P_IN / 101325.0, "atm")
print("Overall phi:", PHI_OVERALL)


## 3. Calculate primary and secondary air

Use a basis of **1 mole of total fuel**.


In [ ]:
n_air_stoic, n_air_primary, n_air_secondary = calculate_air_amounts(0.5, 1.5)
print(f"Stoichiometric air = {n_air_stoic:.4f} mol")
print(f"Primary air        = {n_air_primary:.4f} mol")
print(f"Secondary air      = {n_air_secondary:.4f} mol")


## 4. Simulate one two-stage case

Each case is defined by:

- `x_nh3` — NH₃ mole fraction in the fuel
- `phi_1` — primary-zone equivalence ratio
- `tau` — residence time in seconds

For the initial study:

\[
tau_1=tau_2=tau
\]

The primary reactor is solved first. Its products are then mixed with the calculated secondary air and sent to the secondary reactor.


In [ ]:
TRACKED_SPECIES = ["NH3", "H2", "NO", "NO2", "N2O"]


def get_species_mole_fractions(gas, species_list=TRACKED_SPECIES):
    """Read mole fractions without changing selection; missing names map to zero."""
    species_names = set(gas.species_names)
    mole_fractions = gas.X
    return {
        species: (
            float(mole_fractions[gas.species_index(species)])
            if species in species_names
            else 0.0
        )
        for species in species_list
    }


def simulate_case(params, steady_state_integrator=integrate_to_steady_state):
    """
    Simulate one two-stage NH3/H2 combustion case.

    params = (NH3 mole fraction, primary equivalence ratio, residence time)
    """

    x_nh3, phi_1, tau = params
    x_h2 = 1.0 - x_nh3

    try:
        # =====================================================
        # 1. AIR CALCULATION
        # =====================================================
        n_air_stoic, n_air_1, n_air_2 = calculate_air_amounts(
            x_nh3, phi_1
        )

        if n_air_2 < 0:
            raise ValueError("Secondary air is negative.")

        # =====================================================
        # 2. FUEL COMPOSITION
        # =====================================================
        fuel = {}

        if x_nh3 > 0:
            fuel["NH3"] = x_nh3

        if x_h2 > 0:
            fuel["H2"] = x_h2

        # =====================================================
        # 3. LOAD MECHANISM
        # =====================================================
        gas = ct.Solution(MECH)

        # =====================================================
        # 4. PRIMARY PSR
        # =====================================================
        fuel_stream = ct.Quantity(gas, constant="HP")
        fuel_stream.TPX = T_IN, P_IN, fuel
        fuel_stream.moles = 1.0

        air_stream_1 = ct.Quantity(gas, constant="HP") # cantera quantity object, with constant enthalpy and pressure
        air_stream_1.TPX = T_IN, P_IN, AIR
        air_stream_1.moles = n_air_1

        primary_mix = fuel_stream + air_stream_1

        gas1 = ct.Solution(MECH)
        gas1.TPX = primary_mix.phase.TPX
        gas1.equilibrate("HP")

        reactor1 = ct.IdealGasReactor(gas1, energy="on", clone=False)
        reactor1.volume = 1.0

        inlet1 = ct.Reservoir(primary_mix.phase, clone=False)
        exhaust1 = ct.Reservoir(gas1, clone=False)

        mfc1 = ct.MassFlowController(
            inlet1,
            reactor1,
            mdot=lambda t: reactor1.mass / tau
        )

        pc1 = ct.PressureController(
            reactor1,
            exhaust1,
            primary=mfc1,
            K=0.01
        )

        sim1 = ct.ReactorNet([reactor1])

        converged1, integration_time_1 = steady_state_integrator(
            sim1, reactor1, tau
        )

        if not converged1:
            raise RuntimeError(
                "Primary PSR did not reach steady state after "
                f"{integration_time_1:.3g} s "
                f"({integration_time_1 / tau:.0f} residence times)."
            )

        # Primary outlet quantities.
        primary_gas = reactor1.phase
        primary_X = get_species_mole_fractions(primary_gas)
        T_primary = reactor1.T

        # Transfer the actual primary-reactor outlet state to Stage 2.
        primary_outlet_gas = ct.Solution(MECH) 
        # copy the state of the converged primary reactor to a new gas object for the secondary reactor
        primary_outlet_gas.TDY = reactor1.phase.TDY  # TDY- temperature, density, composition

        primary_products = ct.Quantity(
            primary_outlet_gas,
            constant="HP"
        )
        primary_products.moles = primary_mix.moles # amount of product stream = amount of moles we started with

        # =====================================================
        # 5. SECONDARY / FINAL PSR
        # =====================================================
        air_stream_2 = ct.Quantity(gas, constant="HP")
        air_stream_2.TPX = T_IN, P_IN, AIR
        air_stream_2.moles = n_air_2

        secondary_mix = primary_products + air_stream_2

        overall_phi = n_air_stoic / (n_air_1 + n_air_2)

        gas2 = ct.Solution(MECH)
        gas2.TPX = secondary_mix.phase.TPX

        reactor2 = ct.IdealGasReactor(gas2, energy="on", clone=False)
        reactor2.volume = 1.0

        inlet2 = ct.Reservoir(secondary_mix.phase, clone=False)
        exhaust2 = ct.Reservoir(gas2, clone=False)

        mfc2 = ct.MassFlowController(     # mass flow controller for the secondary reactor
            inlet2,
            reactor2,
            mdot=lambda t: reactor2.mass / tau   #at any simulation time t, the mass flow rate is equal to the mass of the reactor divided by the residence time tau
        )

        pc2 = ct.PressureController(   # pressure controller for the secondary reactor
            reactor2,
            exhaust2,
            primary=mfc2,   # the primary argument specifies that the pressure controller should use the mass flow controller mfc2 to control the flow of gas into the reactor
            K=0.01         # the K argument specifies the proportional gain for the pressure controller, which determines how aggressively it responds to changes in pressure [m(dot)out = m(dot)primary + K*(P(reactor) - P(dowstream))]
        )

        sim2 = ct.ReactorNet([reactor2])    #numerical simulation system that solves its equations

        converged2, integration_time_2 = steady_state_integrator(
            sim2, reactor2, tau
        )  # converged2 is a boolean indicating whether the secondary PSR reached steady state within the specified number of residence times, and integration_time_2 is the total time taken for the integration to reach steady state.

        if not converged2:
            raise RuntimeError(
                "Secondary PSR did not reach steady state after "
                f"{integration_time_2:.3g} s "
                f"({integration_time_2 / tau:.0f} residence times)."
            )

        # Final outlet quantities.
        final_gas = reactor2.phase
        final_X = get_species_mole_fractions(final_gas)

        # =====================================================
        # 6. DERIVED RESULTS
        # =====================================================
        NOx_primary_ppm = (primary_X["NO"] + primary_X["NO2"]) * 1e6
        NOx_final_ppm = (final_X["NO"] + final_X["NO2"]) * 1e6

        # Retain the original notebook's preliminary conversion diagnostic.
        if x_nh3 > 0:
            NH3_conversion = (
                1.0 - final_X["NH3"] * secondary_mix.moles / x_nh3
            ) * 100.0
        else:
            NH3_conversion = np.nan

        if x_h2 > 0:
            H2_conversion = (
                1.0 - final_X["H2"] * secondary_mix.moles / x_h2
            ) * 100.0
        else:
            H2_conversion = np.nan

        return {
            # Operating conditions
            "NH3_frac": x_nh3,
            "H2_frac": x_h2,
            "Phi_1": phi_1,
            "Phi_overall": overall_phi,
            "Tau_ms": tau * 1000.0,

            # Air split
            "Air_stoic": n_air_stoic,
            "Air_primary": n_air_1,
            "Air_secondary": n_air_2,
            "Air_total": n_air_1 + n_air_2,

            # Temperatures
            "T_primary_K": T_primary,
            "T_final_K": reactor2.T,

            # Primary PSR mole fractions
            "X_NH3_primary": primary_X["NH3"],
            "X_H2_primary": primary_X["H2"],
            "X_NO_primary": primary_X["NO"],
            "X_NO2_primary": primary_X["NO2"],
            "X_N2O_primary": primary_X["N2O"],

            # Final PSR mole fractions
            "X_NH3_final": final_X["NH3"],
            "X_H2_final": final_X["H2"],
            "X_NO_final": final_X["NO"],
            "X_NO2_final": final_X["NO2"],
            "X_N2O_final": final_X["N2O"],

            # Primary stage in ppm
            "NO_primary_ppm": primary_X["NO"] * 1e6,
            "NO2_primary_ppm": primary_X["NO2"] * 1e6,
            "NOx_primary_ppm": NOx_primary_ppm,
            "N2O_primary_ppm": primary_X["N2O"] * 1e6,
            "NH3_primary_slip_ppm": primary_X["NH3"] * 1e6,
            "H2_primary_slip_ppm": primary_X["H2"] * 1e6,

            # Final stage in ppm
            "NO_ppm": final_X["NO"] * 1e6,
            "NO2_ppm": final_X["NO2"] * 1e6,
            "NOx_ppm": NOx_final_ppm,
            "N2O_ppm": final_X["N2O"] * 1e6,
            "NH3_slip_ppm": final_X["NH3"] * 1e6,
            "H2_slip_ppm": final_X["H2"] * 1e6,

            # Preliminary conversion diagnostics
            "NH3_conversion": NH3_conversion,
            "H2_conversion": H2_conversion,

            # Numerical information
            "Primary_integration_s": integration_time_1,
            "Secondary_integration_s": integration_time_2,

            "Status": "Converged"
        }

    except Exception as e:
        return {
            "NH3_frac": x_nh3,
            "H2_frac": x_h2,
            "Phi_1": phi_1,
            "Phi_overall": np.nan,
            "Tau_ms": tau * 1000.0,
            "Status": f"FAILED: {type(e).__name__}: {e}"
        }


## 5. Test ONE complete two-stage case

**Run this test before starting the 605-case sweep.**

Test conditions:

- Fuel = 50% NH₃ + 50% H₂
- Primary equivalence ratio φ₁ = 1.5
- Residence time = 60 ms in each PSR
- Overall equivalence ratio = 0.9

### Cantera 3.0.0 compatibility

This notebook uses:

```python
reactor.thermo
```

to access the gas state stored inside an `IdealGasReactor`.

It does **not** use `reactor.phase` and it does **not** call `ReactorNet.solve_steady()`.

If a case fails, the complete Python traceback is printed so that the exact failing line is visible.


In [ ]:
# One complete two-stage test case.
# Do not start the 605-case sweep until this succeeds.

test_case = (
    0.50,   # NH3 fraction in fuel
    1.50,   # Primary equivalence ratio
    0.060   # Residence time [s] = 60 ms
)

print("Running one complete two-stage case...")
result = simulate_case(test_case)

pd.Series(result)


In [ ]:
if result["Status"] == "Converged":
    print("PASS: complete two-stage case converged")
    print(f"Overall phi    = {result['Phi_overall']:.6f}")
    print(f"Primary T      = {result['T_primary_K']:.2f} K")
    print(f"Final T        = {result['T_final_K']:.2f} K")
    print(f"Final NOx      = {result['NOx_ppm']:.3f} ppm")
    print(f"Final N2O      = {result['N2O_ppm']:.3f} ppm")
    print(f"Final NH3 slip = {result['NH3_slip_ppm']:.3f} ppm")
    print(f"Final H2 slip  = {result['H2_slip_ppm']:.3f} ppm")
    print()
    print("Both PSRs were solved by transient integration.")
else:
    print("FAILED")
    print(result["Status"])


## 6. Build the 605-case parameter grid

The full study is:

- 11 NH₃ compositions
- 11 primary φ values
- 5 residence times

\[
11 x 11 x 5=605
\]


In [ ]:
nh3_fractions = np.linspace(0.0, 1.0, 11)

phi_1_values = np.linspace(1.0, 2.0, 11)

residence_times = [
    0.020,  # 20 ms
    0.040,  # 40 ms
    0.060,  # 60 ms
    0.080,  # 80 ms
    0.100   # 100 ms
]

param_grid = list(
    itertools.product(
        nh3_fractions,
        phi_1_values,
        residence_times
    )
)

print("NH3 compositions :", len(nh3_fractions))
print("Primary phi      :", len(phi_1_values))
print("Residence times  :", len(residence_times))
print("Total cases      :", len(param_grid))


## 7. Run the full sweep

This first version runs **serially**. This is intentional: serial execution is much easier to debug.

Once the serial sweep works, multiprocessing can be added.


In [ ]:
start_time = time.time()

results = []

for i, params in enumerate(param_grid, start=1):

    x_nh3, phi_1, tau = params

    print(
        f"[{i:3d}/{len(param_grid)}] "
        f"NH3={x_nh3*100:5.1f}% | "
        f"phi1={phi_1:.1f} | "
        f"tau={tau*1000:.0f} ms",
        end="  "
    )

    result = simulate_case(params)
    results.append(result)

    if result["Status"] == "Converged":
        print(
            f"OK | "
            f"T={result['T_final_K']:.1f} K | "
            f"NOx={result['NOx_ppm']:.2f} ppm"
        )
    else:
        print(result["Status"])

elapsed = time.time() - start_time

print(f"\nSweep finished in {elapsed/60:.2f} minutes.")


## 8. Save results

Every row represents one simulation case.


In [ ]:
df = pd.DataFrame(results)

df.to_csv(
    "results/staged_combustion_results.csv",
    index=False
)

print("Rows:", len(df))
print("Columns:", len(df.columns))

display(df.head())


## 9. Check for failed cases

In [ ]:
failed = df[df["Status"] != "Converged"]

print("Total cases:", len(df))
print("Successful :", len(df) - len(failed))
print("Failed     :", len(failed))

if len(failed) > 0:
    display(
        failed[
            ["NH3_frac", "Phi_1", "Tau_ms", "Status"]
        ]
    )


## 10. Verify overall equivalence ratio

Every successful case should have:

\[
phi_{overall} - 0.9
\]

This is a critical validation check for the secondary-air calculation.


In [ ]:
good = df[df["Status"] == "Converged"].copy()

phi_error = (
    good["Phi_overall"] - PHI_OVERALL
).abs()

print(
    "Maximum |phi_overall - 0.9| =",
    phi_error.max()
)

if phi_error.max() < 1e-10:
    print("PASS: overall equivalence ratio is maintained.")
else:
    print("WARNING: check the air calculation.")


## 11. Prepare data for plotting

The reference presentation studies:

- Temperature
- H₂ mole fraction
- NH₃ mole fraction / NH₃ slip
- NO mole fraction
- NO₂ mole fraction
- N₂O mole fraction
- NOx, where NOx = NO + NO₂

The analysis below stores and plots these quantities at **both the primary
PSR outlet and the final staged-PSR outlet**.

The reference plots mainly use NH₃/H₂ fuel composition on the x-axis and
several primary equivalence ratios as separate curves.


In [ ]:
# Only converged simulations are used for plots.
good = df[df["Status"] == "Converged"].copy()

# Reference residence time for the main comparison plots.
TAU_REF_MS = 60.0

df_tau = good[
    np.isclose(good["Tau_ms"], TAU_REF_MS)
].copy()

# The reference presentation shows phi1 = 1.0 to 1.5 in its curves.
# The full sweep still contains phi1 = 1.0 to 2.0.
PHI_CURVES = [1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0]

print("Converged cases:", len(good))
print("Cases at reference tau:", len(df_tau))


### 11.1 Reusable plotting function

A single plotting function keeps the figures consistent and makes it
easy to add another variable later.


In [ ]:
def plot_vs_fuel_composition(
    data,
    y_column,
    y_label,
    title,
    x_column="NH3_frac",
    x_label="NH₃ mole fraction",
    phi_values=PHI_CURVES,
    marker="o",
    y_scale="linear"
):
    """Plot one result against fuel composition for several phi1 values."""

    fig, ax = plt.subplots(figsize=(8, 6))

    for phi in phi_values:
        sub = data[
            np.isclose(data["Phi_1"], phi)
        ].sort_values(x_column)

        if len(sub) == 0:
            continue

        ax.plot(
            sub[x_column],
            sub[y_column],
            marker=marker,
            label=f"φ₁ = {phi:.1f}"
        )

    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.set_title(title)
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend()

    if y_scale == "log":
        ax.set_yscale("log")

    plt.tight_layout()
    plt.show()


## 12. Final staged-PSR results

These plots correspond most closely to pages 4–5 of the reference
presentation.

The x-axis is NH₃ fraction in the fuel, and each curve represents a
different primary equivalence ratio. Residence time is fixed at 60 ms.


In [ ]:
plot_vs_fuel_composition(
    df_tau,
    y_column="T_final_K",
    y_label="Temperature at final PSR outlet (K)",
    title=f"Final PSR temperature vs NH₃ fraction — τ = {TAU_REF_MS:.0f} ms",
    marker="o",
    y_scale="linear"
)


In [ ]:
plot_vs_fuel_composition(
    df_tau,
    y_column="X_H2_final",
    y_label="H₂ mole fraction at final PSR outlet",
    title=f"Final PSR H₂ mole fraction vs NH₃ fraction — τ = {TAU_REF_MS:.0f} ms",
    marker="v",
    y_scale="linear"
)


In [ ]:
plot_vs_fuel_composition(
    df_tau,
    y_column="X_NO_final",
    y_label="NO mole fraction at final PSR outlet",
    title=f"Final PSR NO mole fraction vs NH₃ fraction — τ = {TAU_REF_MS:.0f} ms",
    marker="s",
    y_scale="linear"
)


In [ ]:
plot_vs_fuel_composition(
    df_tau,
    y_column="X_NO2_final",
    y_label="NO₂ mole fraction at final PSR outlet",
    title=f"Final PSR NO₂ mole fraction vs NH₃ fraction — τ = {TAU_REF_MS:.0f} ms",
    marker="^",
    y_scale="linear"
)


In [ ]:
plot_vs_fuel_composition(
    df_tau,
    y_column="X_N2O_final",
    y_label="N₂O mole fraction at final PSR outlet",
    title=f"Final PSR N₂O mole fraction vs NH₃ fraction — τ = {TAU_REF_MS:.0f} ms",
    marker="D",
    y_scale="linear"
)


In [ ]:
plot_vs_fuel_composition(
    df_tau,
    y_column="X_NH3_final",
    y_label="NH₃ mole fraction at final PSR outlet (NH₃ slip)",
    title=f"Final PSR NH₃ slip vs NH₃ fraction — τ = {TAU_REF_MS:.0f} ms",
    marker="P",
    y_scale="log"
)


### 12.1 Final NOx

NOx is kept as a separate engineering metric:

\[
NO_x = NO + NO_2
\]

N₂O is **not** included in NOx and remains a separate result, matching
the structure of the reference presentation.


In [ ]:
plot_vs_fuel_composition(
    df_tau,
    y_column="NOx_ppm",
    y_label="Final NOx (ppm)",
    title=f"Final NOx vs NH₃ fraction — τ = {TAU_REF_MS:.0f} ms",
    marker="o"
)


## 13. Primary-PSR results

Pages 8–9 of the reference presentation show the corresponding
quantities at the outlet of the **primary PSR**.

These are useful for understanding what enters the secondary-air mixing
stage; they should not be interpreted as final exhaust emissions.


In [ ]:
plot_vs_fuel_composition(
    df_tau,
    y_column="T_primary_K",
    y_label="Temperature at primary PSR outlet (K)",
    title=f"Primary PSR temperature vs NH₃ fraction — τ = {TAU_REF_MS:.0f} ms",
    marker="o",
    y_scale="linear"
)


In [ ]:
plot_vs_fuel_composition(
    df_tau,
    y_column="X_H2_primary",
    y_label="H₂ mole fraction at primary PSR outlet",
    title=f"Primary PSR H₂ mole fraction vs NH₃ fraction — τ = {TAU_REF_MS:.0f} ms",
    marker="v",
    y_scale="linear"
)


In [ ]:
plot_vs_fuel_composition(
    df_tau,
    y_column="X_NO_primary",
    y_label="NO mole fraction at primary PSR outlet",
    title=f"Primary PSR NO mole fraction vs NH₃ fraction — τ = {TAU_REF_MS:.0f} ms",
    marker="s",
    y_scale="linear"
)


In [ ]:
plot_vs_fuel_composition(
    df_tau,
    y_column="X_NO2_primary",
    y_label="NO₂ mole fraction at primary PSR outlet",
    title=f"Primary PSR NO₂ mole fraction vs NH₃ fraction — τ = {TAU_REF_MS:.0f} ms",
    marker="^",
    y_scale="linear"
)


In [ ]:
plot_vs_fuel_composition(
    df_tau,
    y_column="X_N2O_primary",
    y_label="N₂O mole fraction at primary PSR outlet",
    title=f"Primary PSR N₂O mole fraction vs NH₃ fraction — τ = {TAU_REF_MS:.0f} ms",
    marker="D",
    y_scale="linear"
)


In [ ]:
plot_vs_fuel_composition(
    df_tau,
    y_column="X_NH3_primary",
    y_label="NH₃ mole fraction at primary PSR outlet",
    title=f"Primary PSR NH₃ mole fraction vs NH₃ fraction — τ = {TAU_REF_MS:.0f} ms",
    marker="P",
    y_scale="log"
)


## 14. H₂-axis plots

The reference presentation also contains plots with **H₂ mole fraction**
on the x-axis. No additional simulations are required because:

\[
x_{H_2}=1-x_{NH_3}
\]

The same results can therefore be viewed in either fuel-composition
coordinate system.


In [ ]:
def plot_vs_h2_fraction(
    data,
    y_column,
    y_label,
    title,
    phi_values=PHI_CURVES,
    marker="s"
):
    """Plot one result against H2 fuel mole fraction."""

    fig, ax = plt.subplots(figsize=(8, 6))

    for phi in phi_values:
        sub = data[
            np.isclose(data["Phi_1"], phi)
        ].sort_values("H2_frac")

        if len(sub) == 0:
            continue

        ax.plot(
            sub["H2_frac"],
            sub[y_column],
            marker=marker,
            label=f"φ₁ = {phi:.1f}"
        )

    ax.set_xlabel("H₂ mole fraction in fuel")
    ax.set_ylabel(y_label)
    ax.set_title(title)
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend()

    plt.tight_layout()
    plt.show()


plot_vs_h2_fraction(
    df_tau,
    "X_NO_final",
    "NO mole fraction at final PSR outlet",
    f"Final NO vs H₂ fraction — τ = {TAU_REF_MS:.0f} ms"
)

plot_vs_h2_fraction(
    df_tau,
    "X_H2_final",
    "H₂ mole fraction at final PSR outlet",
    f"Final H₂ slip vs H₂ fraction — τ = {TAU_REF_MS:.0f} ms"
)


## 15. Residence-time sensitivity

Residence time is an independent variable in the current 605-case sweep,
so it should be analysed directly as well.

Here the fuel composition and primary equivalence ratio are fixed and
residence time is varied from 20 to 100 ms.


In [ ]:
X_NH3_REF = 0.50
PHI_REF = 1.50

df_tau_sensitivity = good[
    np.isclose(good["NH3_frac"], X_NH3_REF)
    & np.isclose(good["Phi_1"], PHI_REF)
].sort_values("Tau_ms")

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(
    df_tau_sensitivity["Tau_ms"],
    df_tau_sensitivity["T_final_K"],
    marker="o"
)
ax.set_xlabel("Residence time (ms)")
ax.set_ylabel("Final temperature (K)")
ax.set_title("Final temperature vs residence time — 50% NH₃, φ₁ = 1.5")
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(
    df_tau_sensitivity["Tau_ms"],
    df_tau_sensitivity["NOx_ppm"],
    marker="o"
)
ax.set_xlabel("Residence time (ms)")
ax.set_ylabel("Final NOx (ppm)")
ax.set_title("Final NOx vs residence time — 50% NH₃, φ₁ = 1.5")
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(
    df_tau_sensitivity["Tau_ms"],
    df_tau_sensitivity["NH3_slip_ppm"],
    marker="s"
)
ax.set_xlabel("Residence time (ms)")
ax.set_ylabel("Final NH₃ slip (ppm)")
ax.set_title("Final NH₃ slip vs residence time — 50% NH₃, φ₁ = 1.5")
ax.set_yscale("log")
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


## 16. Contour maps

Because the sweep varies both NH₃ fraction and primary equivalence ratio,
contour maps are useful for identifying regions of high/low temperature,
NOx and NH₃ slip.


In [ ]:
def plot_contour(
    data,
    value_column,
    colorbar_label,
    title,
    tau_ms=TAU_REF_MS
):
    """Create an NH3-fraction vs primary-phi contour map."""

    subset = data[
        np.isclose(data["Tau_ms"], tau_ms)
    ].copy()

    pivot = subset.pivot(
        index="Phi_1",
        columns="NH3_frac",
        values=value_column
    )

    X, Y = np.meshgrid(
        pivot.columns.values * 100,
        pivot.index.values
    )

    fig, ax = plt.subplots(figsize=(8, 6))

    contour = ax.contourf(
        X,
        Y,
        pivot.values,
        levels=20
    )

    fig.colorbar(
        contour,
        ax=ax,
        label=colorbar_label
    )

    ax.set_xlabel("NH₃ in fuel (%)")
    ax.set_ylabel("Primary equivalence ratio, φ₁")
    ax.set_title(f"{title} — τ = {tau_ms:.0f} ms")

    plt.tight_layout()
    plt.show()


plot_contour(
    good,
    "T_final_K",
    "Final temperature (K)",
    "Final temperature contour"
)

plot_contour(
    good,
    "NOx_ppm",
    "Final NOx (ppm)",
    "Final NOx contour"
)

plot_contour(
    good,
    "NH3_slip_ppm",
    "Final NH₃ slip (ppm)",
    "Final NH₃ slip contour"
)


## 17. Save the expanded results

The CSV now contains the original operating-condition information plus
the outlet composition of **both PSRs**.

This means the plots can be regenerated later without rerunning the
605 simulations.


In [ ]:
df.to_csv(
    "results/staged_combustion_results_full.csv",
    index=False
)

print("Saved: staged_combustion_results_full.csv")
print("Rows:", len(df))
print("Columns:", len(df.columns))

display(df.head())


# 18. Research-Grade Quantitative Analysis

This section extends the original 605-case two-stage PSR study without changing the original parameter grid or solver definition. It is organized as:

**Parameter effects → Stage-wise changes → Emissions/conversion trade-offs → Sensitivity analysis → Numerical reliability → Non-staged baseline → Reaction-mechanism interpretation**.

All analyses below use the existing `df` results unless an explicitly labelled optional rerun is enabled. Failed simulations are excluded from quantitative response plots unless the analysis is specifically about convergence.

> **Important interpretation rule:** correlations, response surfaces, and rate-of-production rankings are descriptive diagnostics of this 0-D model. They are not, by themselves, proof of causality or of behavior in a spatially resolved combustor.

In [ ]:
# 18.1 Build research metrics from the existing sweep

research = df.copy()

# Convergence flag
research["Converged"] = research["Status"].eq("Converged")
good_research = research[research["Converged"]].copy()

# Stage-wise changes
for species in ["NO", "NO2", "N2O"]:
    research[f"Delta_{species}_ppm"] = (
        research[f"{species}_ppm"] - research[f"{species}_primary_ppm"]
    )

research["Delta_NOx_ppm"] = research["NOx_ppm"] - research["NOx_primary_ppm"]
research["Delta_T_K"] = research["T_final_K"] - research["T_primary_K"]

# Secondary-stage reduction of tracked fuel species.
# This is a stage-to-stage disappearance metric, not an elemental conversion definition.
EPS = 1e-30
research["NH3_stage2_reduction_pct"] = np.where(
    research["X_NH3_primary"] > EPS,
    100.0 * (1.0 - research["X_NH3_final"] / research["X_NH3_primary"]),
    np.nan,
)
research["H2_stage2_reduction_pct"] = np.where(
    research["X_H2_primary"] > EPS,
    100.0 * (1.0 - research["X_H2_final"] / research["X_H2_primary"]),
    np.nan,
)

research["NH3_slip_reduction_pct"] = research["NH3_stage2_reduction_pct"]
research["H2_slip_reduction_pct"] = research["H2_stage2_reduction_pct"]
research["Total_fuel_slip_ppm"] = research["NH3_slip_ppm"] + research["H2_slip_ppm"]
research["Primary_total_fuel_slip_ppm"] = (
    research["NH3_primary_slip_ppm"] + research["H2_primary_slip_ppm"]
)

# Physical air split implied by the specified primary equivalence ratio.
research["Secondary_air_fraction"] = (
    research["Air_secondary"] / research["Air_total"]
)
research["Primary_air_fraction"] = (
    research["Air_primary"] / research["Air_total"]
)

# Relative stage-wise changes; denominator is protected near zero.
research["NOx_stage2_relative_change_pct"] = np.where(
    research["NOx_primary_ppm"].abs() > EPS,
    100.0 * research["Delta_NOx_ppm"] / research["NOx_primary_ppm"],
    np.nan,
)
research["Temperature_rise_pct"] = np.where(
    research["T_primary_K"].abs() > EPS,
    100.0 * research["Delta_T_K"] / research["T_primary_K"],
    np.nan,
)

# Residence-time finite-difference sensitivities within each NH3/phi1 trajectory.
research = research.sort_values(["NH3_frac", "Phi_1", "Tau_ms"]).copy()
for y in ["T_final_K", "NOx_ppm", "NH3_slip_ppm", "H2_slip_ppm"]:
    research[f"d_{y}_d_tau"] = (
        research.groupby(["NH3_frac", "Phi_1"], group_keys=False)[y]
        .diff()
        / research.groupby(["NH3_frac", "Phi_1"], group_keys=False)["Tau_ms"].diff()
    )

# Convenience views used in the remainder of this section.
good_research = research[research["Converged"]].copy()

print(f"Total cases              : {len(research)}")
print(f"Converged cases           : {len(good_research)}")
print(f"Failed cases              : {len(research) - len(good_research)}")
print(f"Convergence rate          : {100*len(good_research)/len(research):.2f}%")
print(f"Max overall-phi error     : {np.nanmax(np.abs(good_research['Phi_overall']-PHI_OVERALL)):.3e}")

summary_metrics = [
    "T_final_K", "Delta_T_K", "NOx_ppm", "Delta_NOx_ppm",
    "NH3_slip_ppm", "H2_slip_ppm", "Total_fuel_slip_ppm",
    "NH3_stage2_reduction_pct", "H2_stage2_reduction_pct"
]
summary_table = good_research[summary_metrics].describe().T[["min", "mean", "max", "std"]]
display(summary_table)

## 19. Parameter effects

The first analysis isolates the observed response to each independent model parameter:

- NH₃ fraction in the fuel, `NH3_frac`
- primary equivalence ratio, `Phi_1`
- residence time, `Tau_ms`

Main-effect plots show the mean response at each parameter level while averaging over the other converged cases. This is a compact way to quantify the sweep without silently treating a nonlinear three-factor system as one-dimensional.

In [ ]:
# 19.1 Main-effect curves with mean ± 1 standard deviation

def main_effect_table(data, parameter, response):
    out = (
        data.groupby(parameter)[response]
        .agg(["mean", "std", "count", "min", "max"])
        .reset_index()
    )
    return out


def plot_main_effects(data, parameter, response, ylabel, title, xscale=None):
    tab = main_effect_table(data, parameter, response)
    fig, ax = plt.subplots(figsize=(8, 5.5))
    x = tab[parameter].to_numpy()
    y = tab["mean"].to_numpy()
    s = tab["std"].fillna(0).to_numpy()
    ax.plot(x, y, marker="o")
    ax.fill_between(x, y-s, y+s, alpha=0.18)
    ax.set_xlabel(parameter)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    if xscale is not None:
        ax.set_xscale(xscale)
    ax.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()
    return tab

# Temperature
plot_main_effects(
    good_research, "NH3_frac", "T_final_K",
    "Final temperature (K)", "Main effect of NH₃ fraction on final temperature"
)
plot_main_effects(
    good_research, "Phi_1", "T_final_K",
    "Final temperature (K)", "Main effect of primary equivalence ratio on final temperature"
)
plot_main_effects(
    good_research, "Tau_ms", "T_final_K",
    "Final temperature (K)", "Main effect of residence time on final temperature"
)

# NOx
plot_main_effects(
    good_research, "NH3_frac", "NOx_ppm",
    "Final NOx (ppm)", "Main effect of NH₃ fraction on final NOx"
)
plot_main_effects(
    good_research, "Phi_1", "NOx_ppm",
    "Final NOx (ppm)", "Main effect of primary equivalence ratio on final NOx"
)
plot_main_effects(
    good_research, "Tau_ms", "NOx_ppm",
    "Final NOx (ppm)", "Main effect of residence time on final NOx"
)

In [ ]:
# 19.2 Air-split interpretation of primary equivalence ratio

air_split_table = (
    good_research[["Phi_1", "Primary_air_fraction", "Secondary_air_fraction"]]
    .drop_duplicates()
    .sort_values("Phi_1")
)
display(air_split_table)

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(air_split_table["Phi_1"], air_split_table["Secondary_air_fraction"], marker="o")
ax.set_xlabel("Primary equivalence ratio, φ₁")
ax.set_ylabel("Secondary-air fraction of total air")
ax.set_title("Air redistribution produced by the staging parameter")
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

## 20. Stage-wise changes

The original notebook reports both PSR outlets. The following quantities directly measure what changes between stages rather than treating the final outlet as the only response:

\[
\Delta T=T_2-T_1
\]
\[
\Delta NO_x=NO_{x,2}-NO_{x,1}
\]

and analogous changes for NO, NO₂, N₂O, NH₃ and H₂.

In [ ]:
# 20.1 Stage-wise change plots at the reference residence time

stage_data = good_research[np.isclose(good_research["Tau_ms"], TAU_REF_MS)].copy()

stage_metrics = [
    ("Delta_T_K", "ΔT across Stage 2 (K)"),
    ("Delta_NOx_ppm", "ΔNOx across Stage 2 (ppm)"),
    ("Delta_NO_ppm", "ΔNO across Stage 2 (ppm)"),
    ("Delta_NO2_ppm", "ΔNO₂ across Stage 2 (ppm)"),
    ("Delta_N2O_ppm", "ΔN₂O across Stage 2 (ppm)"),
    ("NH3_stage2_reduction_pct", "NH₃ stage-2 reduction (%)"),
    ("H2_stage2_reduction_pct", "H₂ stage-2 reduction (%)"),
]

for col, ylabel in stage_metrics:
    plot_vs_fuel_composition(
        stage_data, col, ylabel,
        f"{ylabel} vs NH₃ fraction — τ = {TAU_REF_MS:.0f} ms",
        marker="o", y_scale="linear"
    )

# A direct primary-vs-final comparison for NOx
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(stage_data["NOx_primary_ppm"], stage_data["NOx_ppm"], alpha=0.75)
limit = float(max(stage_data["NOx_primary_ppm"].max(), stage_data["NOx_ppm"].max()) * 1.05)
lims = (0.0, limit)
ax.plot(lims, lims, linestyle="--", linewidth=1)
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel("Primary NOx (ppm)")
ax.set_ylabel("Final NOx (ppm)")
ax.set_title("Primary vs final NOx — τ = 60 ms")
ax.grid(True, linestyle="--", alpha=0.35)
plt.tight_layout(); plt.show()

## 21. Emissions and conversion trade-offs

No single response captures the full engineering behavior. The following plots place final NOx against tracked fuel slip and the reported fuel-conversion diagnostics. A 2-D Pareto screen is included only as a mathematical description of the trade-off; it does not assign an overall optimum.

In [ ]:
# 21.1 Trade-off scatter plots

trade = good_research.copy()

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(
    trade["NOx_ppm"],
    trade["Total_fuel_slip_ppm"],
    c=trade["NH3_frac"],
    cmap="viridis",
    alpha=0.75,
)
fig.colorbar(sc, ax=ax, label="NH₃ fraction in fuel")
ax.set_xlabel("Final NOx (ppm)")
ax.set_ylabel("NH₃ + H₂ slip (ppm)")
ax.set_yscale("log")
ax.set_title("NOx–fuel-slip trade-off across converged cases")
ax.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(
    trade["NOx_ppm"],
    trade["NH3_conversion"],
    c=trade["H2_frac"],
    cmap="plasma",
    alpha=0.75,
)
fig.colorbar(sc, ax=ax, label="H₂ fraction in fuel")
ax.set_xlabel("Final NOx (ppm)")
ax.set_ylabel("Reported NH₃ conversion diagnostic (%)")
ax.set_title("NOx vs reported NH₃ conversion diagnostic")
ax.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(
    trade["NOx_ppm"],
    trade["T_final_K"],
    c=trade["Phi_1"],
    cmap="cividis",
    alpha=0.75,
)
fig.colorbar(sc, ax=ax, label="Primary equivalence ratio, φ₁")
ax.set_xlabel("Final NOx (ppm)")
ax.set_ylabel("Final temperature (K)")
ax.set_title("NOx vs final temperature")
ax.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# 21.2 2-D Pareto screen: minimize NOx and total tracked fuel slip simultaneously

def pareto_efficient_2d(data, x_col="NOx_ppm", y_col="Total_fuel_slip_ppm"):
    vals = data[[x_col, y_col]].to_numpy(dtype=float)
    n = len(vals)
    efficient = np.ones(n, dtype=bool)
    for i in range(n):
        if not np.isfinite(vals[i]).all():
            efficient[i] = False
            continue
        for j in range(n):
            if i == j or not np.isfinite(vals[j]).all():
                continue
            no_worse = (vals[j, 0] <= vals[i, 0]) and (vals[j, 1] <= vals[i, 1])
            strictly_better = (vals[j, 0] < vals[i, 0]) or (vals[j, 1] < vals[i, 1])
            if no_worse and strictly_better:
                efficient[i] = False
                break
    return efficient

trade = trade.reset_index(drop=True)
trade["Pareto_NOx_vs_fuel_slip"] = pareto_efficient_2d(trade)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(
    trade.loc[~trade["Pareto_NOx_vs_fuel_slip"], "NOx_ppm"],
    trade.loc[~trade["Pareto_NOx_vs_fuel_slip"], "Total_fuel_slip_ppm"],
    alpha=0.20,
    label="Dominated"
)
front = trade[trade["Pareto_NOx_vs_fuel_slip"]].sort_values("NOx_ppm")
ax.scatter(front["NOx_ppm"], front["Total_fuel_slip_ppm"], alpha=0.9, label="Pareto-efficient")
ax.set_xlabel("Final NOx (ppm)")
ax.set_ylabel("NH₃ + H₂ slip (ppm)")
ax.set_yscale("log")
ax.set_title("NOx–fuel-slip Pareto screen")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout(); plt.show()

display(
    front.sort_values(["NOx_ppm", "Total_fuel_slip_ppm"])[
        ["NH3_frac", "Phi_1", "Tau_ms", "T_final_K", "NOx_ppm", "NH3_slip_ppm", "H2_slip_ppm"]
    ]
)

## 22. Quantitative sensitivity analysis

Two complementary summaries are calculated:

1. **Main-effect η²**: the fraction of total response variance associated with grouping by one parameter level. This is a descriptive variance-partition measure and does not remove interactions.
2. **Standardized response-surface regression** with linear, quadratic and pairwise-interaction terms.

The regression is evaluated with 5-fold cross-validation using only the converged simulations.

In [ ]:
# 22.1 Main-effect variance fractions (eta-squared)

def eta_squared(data, parameter, response):
    z = data[[parameter, response]].dropna()
    y = z[response].to_numpy(dtype=float)
    grand = np.mean(y)
    ss_total = np.sum((y - grand) ** 2)
    if ss_total == 0:
        return np.nan
    ss_between = 0.0
    for _, g in z.groupby(parameter)[response]:
        ss_between += len(g) * (g.mean() - grand) ** 2
    return ss_between / ss_total

responses_for_sensitivity = [
    "T_final_K", "NOx_ppm", "NH3_slip_ppm", "H2_slip_ppm", "Delta_T_K", "Delta_NOx_ppm"
]
parameters = ["NH3_frac", "Phi_1", "Tau_ms"]

eta_rows = []
for response in responses_for_sensitivity:
    row: dict[str, str | float] = {"Response": response}
    for p in parameters:
        row[f"eta2_{p}"] = eta_squared(good_research, p, response)
    eta_rows.append(row)

eta_table = pd.DataFrame(eta_rows)
display(eta_table)

In [ ]:
# 22.2 Standardized polynomial response-surface regression + 5-fold CV

def standardized_design(data):
    x = data[["NH3_frac", "Phi_1", "Tau_ms"]].to_numpy(dtype=float)
    mu = x.mean(axis=0)
    sd = x.std(axis=0, ddof=0)
    sd[sd == 0] = 1.0
    z = (x - mu) / sd
    X = np.column_stack([
        np.ones(len(z)),
        z[:, 0], z[:, 1], z[:, 2],
        z[:, 0]**2, z[:, 1]**2, z[:, 2]**2,
        z[:, 0]*z[:, 1], z[:, 0]*z[:, 2], z[:, 1]*z[:, 2],
    ])
    names = [
        "Intercept", "NH3_frac", "Phi_1", "Tau_ms",
        "NH3_frac^2", "Phi_1^2", "Tau_ms^2",
        "NH3_frac*Phi_1", "NH3_frac*Tau_ms", "Phi_1*Tau_ms"
    ]
    return X, names


def regression_with_cv(data, response, n_folds=5):
    sub = data.dropna(subset=[response]).reset_index(drop=True)
    X, names = standardized_design(sub)
    y = sub[response].to_numpy(dtype=float)

    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    pred = X @ beta
    ss_res = np.sum((y-pred)**2)
    ss_tot = np.sum((y-y.mean())**2)
    r2 = 1.0 - ss_res/ss_tot if ss_tot > 0 else np.nan
    p = X.shape[1]
    n = len(y)
    adj_r2 = 1.0 - (1.0-r2)*(n-1)/(n-p) if n > p else np.nan

    # Deterministic contiguous folds
    folds = np.array_split(np.arange(n), n_folds)
    cv_sqerr = []
    for test_idx in folds:
        train_mask = np.ones(n, dtype=bool)
        train_mask[test_idx] = False
        b, *_ = np.linalg.lstsq(X[train_mask], y[train_mask], rcond=None)
        yhat = X[test_idx] @ b
        cv_sqerr.extend((y[test_idx] - yhat)**2)
    cv_rmse = float(np.sqrt(np.mean(cv_sqerr))) if cv_sqerr else np.nan

    coef = pd.DataFrame({
        "Term": names,
        "Coefficient": beta,
    })
    summary = pd.Series({
        "Response": response,
        "N": n,
        "R2": r2,
        "Adjusted_R2": adj_r2,
        "CV_RMSE": cv_rmse,
    })
    return summary, coef

regression_summaries = []
regression_coefficients = {}

for response in responses_for_sensitivity:
    summ, coef = regression_with_cv(good_research, response)
    regression_summaries.append(summ)
    regression_coefficients[response] = coef

regression_summary_table = pd.DataFrame(regression_summaries)
display(regression_summary_table)

# Display the full coefficient table for selected key outputs.
for response in ["T_final_K", "NOx_ppm", "NH3_slip_ppm", "Delta_NOx_ppm"]:
    print(f"\n{response}")
    display(regression_coefficients[response])

## 23. Numerical reliability and convergence diagnostics

The original sweep already records failed cases, integration times and the overall-equivalence-ratio check. The following expands that into:

- convergence rate versus each input variable,
- maps of failure locations,
- solver-time distributions,
- and an optional stricter-tolerance rerun of representative cases.

A numerical failure is reported as a numerical event; it is **not** automatically classified as flame extinction, blow-off or another physical instability.

In [ ]:
# 23.1 Convergence rate by parameter level

for parameter in ["NH3_frac", "Phi_1", "Tau_ms"]:
    conv_table = (
        research.groupby(parameter)["Converged"]
        .agg(["mean", "sum", "count"])
        .reset_index()
    )
    conv_table["Convergence_rate_pct"] = 100.0 * conv_table["mean"]
    print(f"\nConvergence by {parameter}")
    display(conv_table[[parameter, "sum", "count", "Convergence_rate_pct"]])

# Failed-case map at 60 ms
failed_research = research[~research["Converged"]].copy()
if len(failed_research) > 0:
    print("Failed cases at reference residence time:")
    display(
        failed_research[np.isclose(failed_research["Tau_ms"], TAU_REF_MS)][
            ["NH3_frac", "Phi_1", "Tau_ms", "Status"]
        ]
    )

    fig, ax = plt.subplots(figsize=(8, 5.5))
    ax.scatter(
        failed_research["NH3_frac"]*100,
        failed_research["Phi_1"],
        s=45
    )
    ax.set_xlabel("NH₃ fraction in fuel (%)")
    ax.set_ylabel("Primary equivalence ratio, φ₁")
    ax.set_title("Locations of failed simulations across the parameter grid")
    ax.grid(True, linestyle="--", alpha=0.35)
    plt.tight_layout(); plt.show()

# Integration-time distributions for converged cases
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.hist(good_research["Primary_integration_s"], bins=25, alpha=0.55, label="Primary")
ax.hist(good_research["Secondary_integration_s"], bins=25, alpha=0.55, label="Secondary")
ax.set_xlabel("Integration time to convergence (s)")
ax.set_ylabel("Number of cases")
ax.set_title("Numerical integration-time distributions")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.25)
plt.tight_layout(); plt.show()

In [ ]:
# 23.2 Optional strict-solver verification
# Default is False to avoid silently launching additional integrations.

RUN_SOLVER_VERIFICATION = False
VERIFY_CASES = [
    (0.50, 1.50, 0.020),
    (0.50, 1.50, 0.060),
    (0.50, 1.50, 0.100),
]


def simulate_case_with_solver_settings(
    params,
    tolerance=1e-9,
    min_residence_times=8.0,
    max_residence_times=150.0,
):
    """Rerun the existing case with stricter convergence settings."""

    def strict_integrator(sim, reactor, tau):
        return integrate_to_steady_state(
            sim,
            reactor,
            tau,
            min_residence_times=min_residence_times,
            max_residence_times=max_residence_times,
            tolerance=tolerance,
        )

    return simulate_case(params, steady_state_integrator=strict_integrator)

if RUN_SOLVER_VERIFICATION:
    verification_rows = []
    for params in VERIFY_CASES:
        default = simulate_case(params)
        strict = simulate_case_with_solver_settings(params)
        row = {
            "NH3_frac": params[0],
            "Phi_1": params[1],
            "Tau_ms": params[2]*1000,
            "Default_status": default["Status"],
            "Strict_status": strict["Status"],
        }
        if default["Status"] == "Converged" and strict["Status"] == "Converged":
            for metric in ["T_final_K", "NOx_ppm", "NH3_slip_ppm", "H2_slip_ppm"]:
                dv, sv = default[metric], strict[metric]
                row[f"{metric}_abs_diff"] = abs(sv-dv)
                row[f"{metric}_rel_diff_pct"] = 100.0 * abs(sv-dv) / max(abs(dv), EPS)
        verification_rows.append(row)
    solver_verification = pd.DataFrame(verification_rows)
    display(solver_verification)
else:
    print("RUN_SOLVER_VERIFICATION=False — set to True to perform representative stricter-tolerance reruns.")

## 24. Comparison with a non-staged PSR baseline

A staged architecture is more informative when compared with a one-stage reference using the **same fuel composition, inlet conditions, overall equivalence ratio and residence time**. The baseline below uses the same Cantera mechanism and the same transient PSR convergence routine.

The full 605-case baseline can be expensive, so the notebook provides three modes:

- `"reference"`: one representative case;
- `"screening"`: a compact 27-case design covering fuel, φ₁ and residence-time endpoints/center;
- `"full"`: all 605 cases.

The default is `"screening"`.

In [ ]:
# 24.1 Single-stage PSR model


def simulate_single_stage_case(params):
    """One-stage PSR with the same inlet fuel, total air, mechanism and residence time."""
    x_nh3, phi_1_unused, tau = params
    x_h2 = 1.0 - x_nh3

    try:
        n_air_stoic, n_air_1, n_air_2 = calculate_air_amounts(x_nh3, phi_1_unused)
        n_air_total = n_air_1 + n_air_2

        fuel = {}
        if x_nh3 > 0:
            fuel["NH3"] = x_nh3
        if x_h2 > 0:
            fuel["H2"] = x_h2

        gas = ct.Solution(MECH)
        fuel_stream = ct.Quantity(gas, constant="HP")
        fuel_stream.TPX = T_IN, P_IN, fuel
        fuel_stream.moles = 1.0

        air_stream = ct.Quantity(gas, constant="HP")
        air_stream.TPX = T_IN, P_IN, AIR
        air_stream.moles = n_air_total

        overall_mix = fuel_stream + air_stream

        gas0 = ct.Solution(MECH)
        gas0.TPX = overall_mix.phase.TPX
        gas0.equilibrate("HP")

        reactor = ct.IdealGasReactor(gas0, energy="on", clone=False)
        reactor.volume = 1.0
        inlet = ct.Reservoir(overall_mix.phase, clone=False)
        exhaust = ct.Reservoir(gas0, clone=False)

        mfc = ct.MassFlowController(
            inlet, reactor,
            mdot=lambda t: reactor.mass / tau
        )
        pc = ct.PressureController(
            reactor, exhaust,
            primary=mfc,
            K=0.01
        )

        sim = ct.ReactorNet([reactor])
        converged, integration_time = integrate_to_steady_state(sim, reactor, tau)
        if not converged:
            raise RuntimeError("Single-stage PSR did not reach steady state.")

        X = get_species_mole_fractions(reactor.phase)
        return {
            "NH3_frac": x_nh3,
            "H2_frac": x_h2,
            "Phi_1": phi_1_unused,
            "Phi_overall": n_air_stoic/n_air_total,
            "Tau_ms": tau*1000,
            "T_single_K": reactor.T,
            "X_NH3_single": X["NH3"],
            "X_H2_single": X["H2"],
            "X_NO_single": X["NO"],
            "X_NO2_single": X["NO2"],
            "X_N2O_single": X["N2O"],
            "NO_single_ppm": X["NO"]*1e6,
            "NO2_single_ppm": X["NO2"]*1e6,
            "NOx_single_ppm": (X["NO"]+X["NO2"])*1e6,
            "N2O_single_ppm": X["N2O"]*1e6,
            "NH3_single_slip_ppm": X["NH3"]*1e6,
            "H2_single_slip_ppm": X["H2"]*1e6,
            "Integration_single_s": integration_time,
            "Status": "Converged",
        }
    except Exception as e:
        return {
            "NH3_frac": x_nh3,
            "H2_frac": x_h2,
            "Phi_1": phi_1_unused,
            "Tau_ms": tau*1000,
            "Status": f"FAILED: {type(e).__name__}: {e}",
        }


BASELINE_MODE = "screening"  # "reference", "screening", or "full"
RUN_SINGLE_STAGE_BASELINE = False

baseline_reference = [(0.50, 1.50, 0.060)]
baseline_screening = list(itertools.product(
    [0.0, 0.5, 1.0],
    [1.0, 1.5, 2.0],
    [0.020, 0.060, 0.100],
))

if BASELINE_MODE == "reference":
    baseline_grid = baseline_reference
elif BASELINE_MODE == "screening":
    baseline_grid = baseline_screening
elif BASELINE_MODE == "full":
    baseline_grid = param_grid
else:
    raise ValueError("BASELINE_MODE must be 'reference', 'screening', or 'full'.")

if RUN_SINGLE_STAGE_BASELINE:
    baseline_results = []
    start_baseline = time.time()
    for i, params in enumerate(baseline_grid, start=1):
        r = simulate_single_stage_case(params)
        baseline_results.append(r)
        if r["Status"] == "Converged":
            print(f"[{i:3d}/{len(baseline_grid)}] NH3={params[0]*100:.0f}% | phi1={params[1]:.1f} | tau={params[2]*1000:.0f} ms | T={r['T_single_K']:.1f} K | NOx={r['NOx_single_ppm']:.2f} ppm")
        else:
            print(f"[{i:3d}/{len(baseline_grid)}] {r['Status']}")

    baseline_df = pd.DataFrame(baseline_results)
    elapsed_baseline = time.time() - start_baseline
    print(f"Baseline finished in {elapsed_baseline/60:.2f} min")

    # Merge using the exact operating conditions.
    staged_for_merge = research[["NH3_frac", "Phi_1", "Tau_ms", "T_final_K", "NOx_ppm", "NH3_slip_ppm", "H2_slip_ppm"]].copy()
    baseline_good = baseline_df[baseline_df["Status"].eq("Converged")].copy()
    comparison_df = staged_for_merge.merge(
        baseline_good,
        on=["NH3_frac", "Phi_1", "Tau_ms"],
        how="inner"
    )

    comparison_df["Delta_T_staged_minus_single_K"] = comparison_df["T_final_K"] - comparison_df["T_single_K"]
    comparison_df["Delta_NOx_staged_minus_single_ppm"] = comparison_df["NOx_ppm"] - comparison_df["NOx_single_ppm"]
    comparison_df["Delta_NH3_slip_staged_minus_single_ppm"] = comparison_df["NH3_slip_ppm"] - comparison_df["NH3_single_slip_ppm"]
    comparison_df["Delta_H2_slip_staged_minus_single_ppm"] = comparison_df["H2_slip_ppm"] - comparison_df["H2_single_slip_ppm"]
    comparison_df["Delta_total_fuel_slip_staged_minus_single_ppm"] = (
        comparison_df["NH3_slip_ppm"] + comparison_df["H2_slip_ppm"]
        - comparison_df["NH3_single_slip_ppm"] - comparison_df["H2_single_slip_ppm"]
    )

    display(comparison_df.head(20))

    if len(comparison_df) > 0:
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.scatter(comparison_df["NOx_single_ppm"], comparison_df["NOx_ppm"], alpha=0.8)
        lims = [0, max(comparison_df["NOx_single_ppm"].max(), comparison_df["NOx_ppm"].max()) * 1.05]
        ax.plot(lims, lims, linestyle="--", linewidth=1)
        ax.set_xlabel("Single-stage NOx (ppm)")
        ax.set_ylabel("Two-stage NOx (ppm)")
        ax.set_title(f"Two-stage vs single-stage NOx — baseline mode: {BASELINE_MODE}")
        ax.grid(True, linestyle="--", alpha=0.3)
        plt.tight_layout(); plt.show()

        fig, ax = plt.subplots(figsize=(8, 6))
        ax.scatter(comparison_df["T_single_K"], comparison_df["T_final_K"], alpha=0.8)
        lims = [min(comparison_df["T_single_K"].min(), comparison_df["T_final_K"].min())*0.995,
                max(comparison_df["T_single_K"].max(), comparison_df["T_final_K"].max())*1.005]
        ax.plot(lims, lims, linestyle="--", linewidth=1)
        ax.set_xlabel("Single-stage temperature (K)")
        ax.set_ylabel("Two-stage final temperature (K)")
        ax.set_title(f"Two-stage vs single-stage temperature — baseline mode: {BASELINE_MODE}")
        ax.grid(True, linestyle="--", alpha=0.3)
        plt.tight_layout(); plt.show()
else:
    print(f"RUN_SINGLE_STAGE_BASELINE=False — set to True to run the {BASELINE_MODE} non-staged comparison ({len(baseline_grid)} cases).")

## 25. Reaction-mechanism interpretation

Outlet concentrations show **what** changes; a local rate-of-production (ROP) analysis helps examine **which reactions contribute to the instantaneous formation or consumption of a species at a selected PSR outlet**.

The following optional analysis reruns a selected case and retains the full Cantera gas objects. It then ranks reactions by their contribution to the net production rate of NO, NH₃ and H₂. These rankings are local steady-state diagnostics, not integrated pathway fractions.

In [ ]:
# 25.1 Detailed two-stage rerun retaining full gas states


def solve_detailed_two_stage_case(params):
    """Rerun one two-stage case and retain full Cantera outlet states for ROP analysis."""
    x_nh3, phi_1, tau = params
    x_h2 = 1.0 - x_nh3

    n_air_stoic, n_air_1, n_air_2 = calculate_air_amounts(x_nh3, phi_1)
    fuel = {}
    if x_nh3 > 0:
        fuel["NH3"] = x_nh3
    if x_h2 > 0:
        fuel["H2"] = x_h2

    gas = ct.Solution(MECH)
    fuel_stream = ct.Quantity(gas, constant="HP")
    fuel_stream.TPX = T_IN, P_IN, fuel
    fuel_stream.moles = 1.0

    air_stream_1 = ct.Quantity(gas, constant="HP")
    air_stream_1.TPX = T_IN, P_IN, AIR
    air_stream_1.moles = n_air_1
    primary_mix = fuel_stream + air_stream_1

    gas1 = ct.Solution(MECH)
    gas1.TPX = primary_mix.phase.TPX
    gas1.equilibrate("HP")
    reactor1 = ct.IdealGasReactor(gas1, energy="on", clone=False)
    reactor1.volume = 1.0
    inlet1 = ct.Reservoir(primary_mix.phase, clone=False)
    exhaust1 = ct.Reservoir(gas1, clone=False)
    mfc1 = ct.MassFlowController(inlet1, reactor1, mdot=lambda t: reactor1.mass/tau)
    pc1 = ct.PressureController(reactor1, exhaust1, primary=mfc1, K=0.01)
    sim1 = ct.ReactorNet([reactor1])
    ok1, time1 = integrate_to_steady_state(sim1, reactor1, tau)
    if not ok1:
        raise RuntimeError("Primary detailed rerun did not converge")

    primary_state = ct.Solution(MECH)
    primary_state.TDY = reactor1.phase.TDY
    primary_products = ct.Quantity(primary_state, constant="HP")
    primary_products.moles = primary_mix.moles

    gas_air2 = ct.Solution(MECH)
    air_stream_2 = ct.Quantity(gas_air2, constant="HP")
    air_stream_2.TPX = T_IN, P_IN, AIR
    air_stream_2.moles = n_air_2
    secondary_mix = primary_products + air_stream_2

    gas2 = ct.Solution(MECH)
    gas2.TPX = secondary_mix.phase.TPX
    reactor2 = ct.IdealGasReactor(gas2, energy="on", clone=False)
    reactor2.volume = 1.0
    inlet2 = ct.Reservoir(secondary_mix.phase, clone=False)
    exhaust2 = ct.Reservoir(gas2, clone=False)
    mfc2 = ct.MassFlowController(inlet2, reactor2, mdot=lambda t: reactor2.mass/tau)
    pc2 = ct.PressureController(reactor2, exhaust2, primary=mfc2, K=0.01)
    sim2 = ct.ReactorNet([reactor2])
    ok2, time2 = integrate_to_steady_state(sim2, reactor2, tau)
    if not ok2:
        raise RuntimeError("Secondary detailed rerun did not converge")

    final_state = ct.Solution(MECH)
    final_state.TDY = reactor2.phase.TDY

    return {
        "primary_gas": primary_state,
        "final_gas": final_state,
        "primary_integration_s": time1,
        "final_integration_s": time2,
    }


def reaction_contribution_table(gas, species_name, top_n=15):
    """Local reaction contribution to the net production rate of one species."""
    k = gas.species_index(species_name)
    net_rop = gas.net_rates_of_progress
    nu = gas.product_stoich_coeffs[k, :] - gas.reactant_stoich_coeffs[k, :]
    contribution = nu * net_rop

    table = pd.DataFrame({
        "Reaction_index": np.arange(gas.n_reactions),
        "Reaction": [gas.reaction(i).equation for i in range(gas.n_reactions)],
        "Contribution_kmol_m3_s": contribution,
        "Abs_contribution": np.abs(contribution),
    })
    table["Sign"] = np.where(table["Contribution_kmol_m3_s"] >= 0, "formation", "consumption")
    positive = table[table["Contribution_kmol_m3_s"] > 0].nlargest(top_n, "Contribution_kmol_m3_s")
    negative = table[table["Contribution_kmol_m3_s"] < 0].nsmallest(top_n, "Contribution_kmol_m3_s")
    return positive, negative, table


RUN_ROP_ANALYSIS = False
ROP_CASE = (0.50, 1.50, 0.060)
ROP_TOP_N = 12

if RUN_ROP_ANALYSIS:
    detail = solve_detailed_two_stage_case(ROP_CASE)
    for stage_name, gas_state in [("Primary", detail["primary_gas"]), ("Final", detail["final_gas"])]:
        print(f"\n===== {stage_name} stage: NO rate of production =====")
        pos_no, neg_no, _ = reaction_contribution_table(gas_state, "NO", top_n=ROP_TOP_N)
        display(pos_no[["Reaction_index", "Reaction", "Contribution_kmol_m3_s"]])
        display(neg_no[["Reaction_index", "Reaction", "Contribution_kmol_m3_s"]])

        print(f"===== {stage_name} stage: NH3 rate of production =====")
        pos_nh3, neg_nh3, _ = reaction_contribution_table(gas_state, "NH3", top_n=ROP_TOP_N)
        display(pos_nh3[["Reaction_index", "Reaction", "Contribution_kmol_m3_s"]])
        display(neg_nh3[["Reaction_index", "Reaction", "Contribution_kmol_m3_s"]])

        # Visualize the largest absolute contributions.
        _, _, full_no = reaction_contribution_table(gas_state, "NO", top_n=ROP_TOP_N)
        plot_no = pd.concat([
            full_no[full_no["Contribution_kmol_m3_s"] > 0].nlargest(ROP_TOP_N, "Contribution_kmol_m3_s"),
            full_no[full_no["Contribution_kmol_m3_s"] < 0].nsmallest(ROP_TOP_N, "Contribution_kmol_m3_s"),
        ]).sort_values("Contribution_kmol_m3_s")

        fig, ax = plt.subplots(figsize=(10, 7))
        ax.barh(np.arange(len(plot_no)), plot_no["Contribution_kmol_m3_s"])
        ax.set_yticks(np.arange(len(plot_no)))
        ax.set_yticklabels(plot_no["Reaction"], fontsize=8)
        ax.set_xlabel("Contribution to local NO production rate (kmol m⁻³ s⁻¹)")
        ax.set_title(f"Top local NO reaction contributions — {stage_name} stage")
        ax.grid(True, axis="x", linestyle="--", alpha=0.3)
        plt.tight_layout(); plt.show()
else:
    print("RUN_ROP_ANALYSIS=False — set to True to rerun the reference case and perform local reaction-rate-of-production analysis.")

## 26. Quantitative research summary and export

The tables below collect the principal ranges and sensitivity diagnostics used in the report. They are intended to make the notebook directly reusable for the Results and Discussion chapters.

In [ ]:
# 26.1 Compact quantitative summary

report_summary = pd.DataFrame({
    "Metric": [
        "Total simulated cases",
        "Converged cases",
        "Failed cases",
        "Convergence rate (%)",
        "Final temperature (K)",
        "Stage-2 temperature rise (K)",
        "Final NOx (ppm)",
        "Stage-2 NOx change (ppm)",
        "Final NH3 slip (ppm)",
        "Final H2 slip (ppm)",
    ],
    "Minimum": [
        len(research), len(good_research), len(research)-len(good_research),
        100*len(good_research)/len(research),
        good_research["T_final_K"].min(),
        good_research["Delta_T_K"].min(),
        good_research["NOx_ppm"].min(),
        good_research["Delta_NOx_ppm"].min(),
        good_research["NH3_slip_ppm"].min(),
        good_research["H2_slip_ppm"].min(),
    ],
    "Mean": [
        np.nan, np.nan, np.nan, 100*len(good_research)/len(research),
        good_research["T_final_K"].mean(),
        good_research["Delta_T_K"].mean(),
        good_research["NOx_ppm"].mean(),
        good_research["Delta_NOx_ppm"].mean(),
        good_research["NH3_slip_ppm"].mean(),
        good_research["H2_slip_ppm"].mean(),
    ],
    "Maximum": [
        len(research), len(good_research), len(research)-len(good_research),
        100*len(good_research)/len(research),
        good_research["T_final_K"].max(),
        good_research["Delta_T_K"].max(),
        good_research["NOx_ppm"].max(),
        good_research["Delta_NOx_ppm"].max(),
        good_research["NH3_slip_ppm"].max(),
        good_research["H2_slip_ppm"].max(),
    ],
})
display(report_summary)

# Save only data generated by the notebook; no source-model files are altered.
research.to_csv("results/staged_combustion_research_metrics.csv", index=False)
eta_table.to_csv("results/staged_combustion_eta_squared.csv", index=False)
regression_summary_table.to_csv("results/staged_combustion_regression_summary.csv", index=False)

print("Saved research metric tables:")
print("  staged_combustion_research_metrics.csv")
print("  staged_combustion_eta_squared.csv")
print("  staged_combustion_regression_summary.csv")

### Interpretation checklist for the written report

When converting these outputs into the BTP report, report numerical observations in the following order:

1. **Parameter effect:** state the direction and magnitude of the observed response over the simulated range.
2. **Stage effect:** quantify the change from primary to final PSR (`ΔT`, `ΔNOx`, species-slip reduction).
3. **Trade-off:** report NOx together with fuel slip/conversion rather than treating one response in isolation.
4. **Sensitivity:** use η² and the standardized response-surface coefficients to identify influential parameters and interactions, while explicitly noting that these are model-based statistical diagnostics.
5. **Reliability:** report the 586/605 convergence result, the overall-φ error, failed-case locations and any stricter-solver verification that was actually run.
6. **Baseline:** compare staged and single-stage simulations at matched operating conditions before attributing an effect specifically to staging.
7. **Mechanism:** use local Cantera rate-of-production results only to support mechanistic hypotheses; do not describe them as integrated pathway fractions unless a dedicated pathway integration is performed.